## Week 5 Day 1: Agent Foundations
### Reasoning Loops, Tool Calling & Raw Python Agents

**Objective:** Build a minimal AI agent from scratch using Groq API, understanding the core reasoning loop before touching frameworks like LangChain or LangGraph.

**Key Concepts:**
- What makes an "agent" agentic: autonomy, tool use, multi-step planning, self-correction
- ReAct pattern: Reason → Act → Observe → repeat
- Tool calling and function execution
- Agent state management and memory
- Failure modes and guardrails

---

## Section 1: Setup & Environment Configuration

Load dependencies and configure Groq API client using credentials from .env file.

In [5]:
import os
import json
import re
import time
from typing import Any, Optional
from dotenv import load_dotenv
from groq import Groq

# Load environment variables from .env file
load_dotenv()

# Initialize Groq client with API key from .env
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY not found in .env file. Please add it.")

client = Groq(api_key=GROQ_API_KEY)

print("Environment loaded")
print(f"Groq API key configured (length: {len(GROQ_API_KEY)} chars)")
print(f"Using Groq API with client instance: {type(client).__name__}")

Environment loaded
Groq API key configured (length: 56 chars)
Using Groq API with client instance: Groq


In [11]:
# Check available models on your Groq account
print("\nChecking available models on your Groq account...")
try:
    models = client.models.list()
    available_models = [model.id for model in models.data]
    print(f"Available models ({len(available_models)}):")
    for model in available_models:
        print(f"  • {model}")
    
    # Choose a good model for tool calling
    if any("70b" in m for m in available_models):
        AGENT_MODEL = [m for m in available_models if "70b" in m][0]
    elif any("8b" in m for m in available_models):
        AGENT_MODEL = [m for m in available_models if "8b" in m][0]
    else:
        AGENT_MODEL = available_models[0]
    
    print(f"\n✓ Selected model for agents: {AGENT_MODEL}")
except Exception as e:
    print(f"Could not list models, using default: {str(e)}")
    AGENT_MODEL = "gemma-7b-it"  # Fallback


Checking available models on your Groq account...
Available models (15):
  • meta-llama/llama-prompt-guard-2-86m
  • groq/compound
  • llama-3.1-8b-instant
  • canopylabs/orpheus-v1-english
  • groq/compound-mini
  • openai/gpt-oss-120b
  • whisper-large-v3
  • openai/gpt-oss-20b
  • meta-llama/llama-prompt-guard-2-22m
  • openai/gpt-oss-safeguard-20b
  • whisper-large-v3-turbo
  • canopylabs/orpheus-arabic-saudi
  • allam-2-7b
  • llama-3.3-70b-versatile
  • qwen/qwen3.6-27b

✓ Selected model for agents: llama-3.3-70b-versatile


## Section 2: Agent Concepts & Mental Model

### Agent vs. Chatbot vs. Workflow

**Chatbot:**
- Receives user input → responds based on training/rules
- No memory of past decisions beyond current conversation
- Single turn-oriented (user input → bot output)
- Example: "What's the weather?" → predefined response

**Workflow:**
- A fixed sequence of steps executed in order
- No branching based on intermediate results
- Example: "Step 1: fetch data → Step 2: process → Step 3: save"

**Agent:**
- LLM in a loop that reasons, chooses actions, observes results, and adjusts
- Autonomy: makes decisions about which tools to call and when
- Multi-step planning: breaks complex tasks into sub-tasks
- Self-correction: adapts based on tool feedback
- Example: "Analyze weather in 2 cities, compare, and recommend best one"

### What Makes Something "Agentic"?

1. **Autonomy**: Agent decides what to do next, not following a rigid script
2. **Tool Use**: Can invoke external tools/functions based on reasoning
3. **Multi-step Planning**: Decomposes complex goals into sub-tasks
4. **Self-Correction**: Observes tool results and adapts strategy if needed
5. **Reasoning**: Explicitly states "why" before taking action

### ReAct Pattern (Reason → Act → Observe → Repeat)

```
┌─────────────────────────────────────┐
│   USER GOAL / AGENT INIT            │
└──────────────┬──────────────────────┘
               ▼
         ┌──────────────┐
         │   REASON     │  (LLM thinks about what to do)
         │ "I need to   │
         │  call X tool"│
         └──────┬───────┘
                ▼
         ┌──────────────┐
         │     ACT      │  (Execute the chosen tool)
         │  Call tool() │
         └──────┬───────┘
                ▼
         ┌──────────────┐
         │   OBSERVE    │  (Receive and log result)
         │ Get result   │
         └──────┬───────┘
                ▼
         ┌──────────────┐
         │  Is task     │
         │  complete?   │
         └──────┬───────┘
            YES │ NO
         ┌──────┴────────┐
         ▼               ▼
    RETURN ANSWER   (REASON again)
```

### ReAct Pseudocode

```python
while task_not_complete and iterations < max_iterations:
    # REASON: LLM decides what to do
    response = llm.think(messages)
    
    # Check if tool call needed
    if response.has_tool_call():
        tool_name, args = response.extract_tool()
        
        # ACT: Execute the tool
        result = execute_tool(tool_name, args)
        
        # OBSERVE: Add result back to message history
        messages.append({"role": "assistant", "content": response})
        messages.append({"role": "user", "content": f"Tool {tool_name} returned: {result}"})
    else:
        # No more tools needed - RETURN ANSWER
        return response.final_text()
```

### When Is An Agent Overkill?

Use a **simple prompt or script** when:
- The task is single-step and deterministic (e.g., "translate this text")
- The LLM can answer directly without external tools
- The task has no branching logic or decision-making

Examples of overkill: 
- "Summarize this PDF" → Just send the text to the LLM
- "Convert CSV to JSON" → Write a simple Python script
- "Generate 10 random names" → Direct LLM call

Examples where agent is necessary: 
- "Find the 3 richest people in dataset X and send them an email"
- "Monitor 5 stock prices and alert when any drops 10%"
- "Research [topic] and write a report, including external links"

## Section 3: Tool Calling Fundamentals

Define tools with proper JSON schemas. The Groq API (via Claude backend) supports function calling where tools are passed as structured definitions. Quality descriptions help the model choose the right tool.

In [6]:
# Define tool schemas with proper descriptions
tools = [
    {
        "type": "function",
        "function": {
            "name": "calculator",
            "description": "Performs basic arithmetic operations (add, subtract, multiply, divide). Use this when you need to compute numerical results.",
            "parameters": {
                "type": "object",
                "properties": {
                    "operation": {
                        "type": "string",
                        "enum": ["add", "subtract", "multiply", "divide"],
                        "description": "The arithmetic operation to perform"
                    },
                    "operand_a": {
                        "type": "number",
                        "description": "The first number"
                    },
                    "operand_b": {
                        "type": "number",
                        "description": "The second number"
                    }
                },
                "required": ["operation", "operand_a", "operand_b"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Retrieves current weather information for a given city. Returns temperature in Celsius and weather condition.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "Name of the city (e.g., 'London', 'Tokyo')"
                    },
                    "units": {
                        "type": "string",
                        "enum": ["celsius", "fahrenheit"],
                        "description": "Temperature unit to return. Defaults to celsius."
                    }
                },
                "required": ["city"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "read_file",
            "description": "Reads and returns the contents of a text file from the local filesystem.",
            "parameters": {
                "type": "object",
                "properties": {
                    "file_path": {
                        "type": "string",
                        "description": "Path to the file to read (e.g., 'data.txt', 'logs/output.txt')"
                    }
                },
                "required": ["file_path"]
            }
        }
    }
]

print("Tool schemas defined:")
for tool in tools:
    func = tool["function"]
    print(f"\n  • {func['name']}: {func['description']}")
    print(f"    Params: {list(func['parameters']['properties'].keys())}")

Tool schemas defined:

  • calculator: Performs basic arithmetic operations (add, subtract, multiply, divide). Use this when you need to compute numerical results.
    Params: ['operation', 'operand_a', 'operand_b']

  • get_weather: Retrieves current weather information for a given city. Returns temperature in Celsius and weather condition.
    Params: ['city', 'units']

  • read_file: Reads and returns the contents of a text file from the local filesystem.
    Params: ['file_path']


In [7]:
# Tool implementation functions

def execute_calculator(operation: str, operand_a: float, operand_b: float) -> str:
    """Execute calculator operation and return result."""
    try:
        if operation == "add":
            result = operand_a + operand_b
        elif operation == "subtract":
            result = operand_a - operand_b
        elif operation == "multiply":
            result = operand_a * operand_b
        elif operation == "divide":
            if operand_b == 0:
                return "Error: Cannot divide by zero"
            result = operand_a / operand_b
        else:
            return f"Error: Unknown operation '{operation}'"
        
        return f"{operand_a} {operation} {operand_b} = {result}"
    except Exception as e:
        return f"Error executing calculator: {str(e)}"


def execute_get_weather(city: str, units: str = "celsius") -> str:
    """Mock weather lookup - returns simulated weather data."""
    # Simulated weather data
    weather_data = {
        "london": {"celsius": 12, "fahrenheit": 54, "condition": "Rainy"},
        "tokyo": {"celsius": 18, "fahrenheit": 64, "condition": "Partly Cloudy"},
        "new york": {"celsius": 15, "fahrenheit": 59, "condition": "Sunny"},
        "dubai": {"celsius": 38, "fahrenheit": 100, "condition": "Hot & Clear"},
        "sydney": {"celsius": 22, "fahrenheit": 72, "condition": "Sunny"}
    }
    
    city_lower = city.lower()
    if city_lower in weather_data:
        data = weather_data[city_lower]
        temp = data[units]
        condition = data["condition"]
        return f"Weather in {city}: {temp}° {units.capitalize()}, {condition}"
    else:
        return f"Error: Weather data not available for {city}"


def execute_read_file(file_path: str) -> str:
    """Read file from filesystem."""
    try:
        if not os.path.exists(file_path):
            return f"Error: File not found: {file_path}"
        
        with open(file_path, 'r', encoding='utf-8') as f:
            content = f.read()
        
        return f"File contents ({len(content)} chars):\n{content[:500]}..."  # Return first 500 chars
    except Exception as e:
        return f"Error reading file: {str(e)}"


def execute_tool(tool_name: str, tool_input: dict) -> str:
    """Router function to execute the appropriate tool."""
    print(f"\nExecuting tool: {tool_name}")
    print(f"     Input: {tool_input}")
    
    if tool_name == "calculator":
        result = execute_calculator(
            tool_input.get("operation"),
            tool_input.get("operand_a"),
            tool_input.get("operand_b")
        )
    elif tool_name == "get_weather":
        result = execute_get_weather(
            tool_input.get("city"),
            tool_input.get("units", "celsius")
        )
    elif tool_name == "read_file":
        result = execute_read_file(tool_input.get("file_path"))
    else:
        result = f"Error: Unknown tool '{tool_name}'"
    
    print(f"     Result: {result}")
    return result

print("Tool implementations defined")

Tool implementations defined


### Single Tool Call Example

Send a request where the model chooses a tool, manually execute it, and return the result.

In [16]:
# Step 1: Send initial request with tool definitions
print("=" * 60)
print("SINGLE TOOL CALL EXAMPLE")
print("=" * 60)

user_query = "What is the weather in London and Tokyo? Which is warmer?"
print(f"\nUser Query: {user_query}\n")

messages = [
    {"role": "user", "content": user_query}
]

# Send to Groq with tool definitions
print(f"Step 1: Sending request to Groq with tool definitions (model: {AGENT_MODEL})...")
response = client.chat.completions.create(
    model=AGENT_MODEL,
    messages=messages,
    tools=tools,
    tool_choice="auto",  # Let model decide if it needs tools
    temperature=0.7,
    max_tokens=1024
)

print(f"Response received.")
print(f"Stop reason: {response.choices[0].finish_reason}")

# Step 2: Check if model called a tool
assistant_message = response.choices[0].message
print(f"\nAssistant response type: {type(assistant_message).__name__}")

# Access tool calls
if hasattr(assistant_message, 'tool_calls') and assistant_message.tool_calls:
    print(f"\nModel made {len(assistant_message.tool_calls)} tool call(s)")
    
    for idx, tool_call in enumerate(assistant_message.tool_calls, 1):
        tool_name = tool_call.function.name
        tool_input = json.loads(tool_call.function.arguments)
        
        print(f"\nTool Call {idx}:")
        print(f"  Name: {tool_name}")
        print(f"  Input: {tool_input}")
        
        # Step 3: Manually execute the tool
        print("\nStep 2: Manually executing tool...")
        result = execute_tool(tool_name, tool_input)
        
        # Step 4: Build tool_result message block
        print("\nStep 3: Building tool_result message block...")
        messages.append({"role": "assistant", "content": assistant_message.content or ""})
        
        # For Groq/Anthropic format, append tool result
        tool_result_content = f"Tool {tool_name} executed successfully. Result: {result}"
        messages.append({
            "role": "user",
            "content": tool_result_content
        })
        
        print(f"Tool result added to message history")
        print(f"Content: {tool_result_content[:80]}...")

else:
    print("\nModel did not call any tools in this response")
    if assistant_message.content:
        print(f"Direct response: {assistant_message.content}")

SINGLE TOOL CALL EXAMPLE

User Query: What is the weather in London and Tokyo? Which is warmer?

Step 1: Sending request to Groq with tool definitions (model: llama-3.3-70b-versatile)...
Response received.
Stop reason: tool_calls

Assistant response type: ChatCompletionMessage

Model made 2 tool call(s)

Tool Call 1:
  Name: get_weather
  Input: {'city': 'London'}

Step 2: Manually executing tool...

Executing tool: get_weather
     Input: {'city': 'London'}
     Result: Weather in London: 12° Celsius, Rainy

Step 3: Building tool_result message block...
Tool result added to message history
Content: Tool get_weather executed successfully. Result: Weather in London: 12° Celsius, ...

Tool Call 2:
  Name: get_weather
  Input: {'city': 'Tokyo'}

Step 2: Manually executing tool...

Executing tool: get_weather
     Input: {'city': 'Tokyo'}
     Result: Weather in Tokyo: 18° Celsius, Partly Cloudy

Step 3: Building tool_result message block...
Tool result added to message history
Content: To

## Section 4: Build a Minimal Agent Loop

Implement a while-loop agent that:
1. Sends message to LLM with tool definitions
2. Checks if response contains tool_use
3. Executes tool if needed
4. Appends tool_result to message history
5. Repeats until model returns final text answer

Includes max_iterations safeguard and error handling.

In [ ]:
class MinimalAgent:
    """A minimal agentic loop using Groq API."""
    
    def __init__(self, max_iterations: int = 10, debug: bool = True, model: str = None):
        self.max_iterations = max_iterations
        self.debug = debug
        self.iteration_count = 0
        self.messages = []
        self.model = model or AGENT_MODEL
    
    def log(self, message: str, level: str = "INFO"):
        """Debug logging helper."""
        if self.debug:
            prefix = "  " * self.iteration_count
            print(f"{prefix}[{level}] {message}")
    
    def run(self, user_query: str) -> str:
        """Run the agent loop until completion."""
        self.iteration_count = 0
        self.messages = [{"role": "user", "content": user_query}]
        
        self.log(f"Agent starting with query: {user_query}")
        print()
        
        while self.iteration_count < self.max_iterations:
            self.iteration_count += 1
            self.log(f"Iteration {self.iteration_count} of {self.max_iterations}")
            
            try:
                # REASON: Send to LLM with tool definitions
                self.log("Sending request to LLM...")
                response = client.chat.completions.create(
                    model=self.model,
                    messages=self.messages,
                    tools=tools,
                    tool_choice="auto",
                    temperature=0.7,
                    max_tokens=1024
                )
                
                assistant_message = response.choices[0].message
                finish_reason = response.choices[0].finish_reason
                
                self.log(f"LLM responded (finish_reason: {finish_reason})")
                
                # Check if model called any tools
                if hasattr(assistant_message, 'tool_calls') and assistant_message.tool_calls:
                    # ACT: Execute tools
                    self.log(f"Model called {len(assistant_message.tool_calls)} tool(s)")
                    
                    # Add assistant message to history
                    self.messages.append({
                        "role": "assistant",
                        "content": assistant_message.content or ""
                    })
                    
                    # Execute each tool call
                    for tool_call in assistant_message.tool_calls:
                        tool_name = tool_call.function.name
                        tool_input = json.loads(tool_call.function.arguments)
                        
                        self.log(f"  Executing: {tool_name}({tool_input})")
                        
                        try:
                            # OBSERVE: Execute tool and get result
                            result = execute_tool(tool_name, tool_input)
                            self.log(f"Tool returned: {result[:80]}...")
                            
                            # Append tool result to messages
                            self.messages.append({
                                "role": "user",
                                "content": f"Tool {tool_name} result: {result}"
                            })
                        except Exception as e:
                            error_msg = f"Tool execution failed: {str(e)}"
                            self.log(f"  ✗ {error_msg}", level="ERROR")
                            self.messages.append({
                                "role": "user",
                                "content": f"Error executing {tool_name}: {str(e)}"
                            })
                
                else:
                    # No more tools - return final answer
                    final_answer = assistant_message.content or "No response generated"
                    self.log(f"Agent complete. Final answer:")
                    print(f"\n{'='*60}")
                    print(f"FINAL ANSWER (Iteration {self.iteration_count}):")
                    print(f"{'='*60}")
                    print(final_answer)
                    print(f"{'='*60}\n")
                    
                    return final_answer
                    
            except Exception as e:
                self.log(f"Error in agent loop: {str(e)}", level="ERROR")
                return f"Agent failed: {str(e)}"
        
        # Reached max iterations
        error_msg = f"Agent exceeded max_iterations ({self.max_iterations})"
        self.log(error_msg, level="ERROR")
        return error_msg

print("MinimalAgent class defined")

MinimalAgent class defined


### Test: Multi-Step Task

Test agent on a task requiring 2+ tool calls: "Look up the weather in London and Tokyo, then tell me which is warmer."

In [21]:
# Initialize and run agent
agent = MinimalAgent(max_iterations=10, debug=True)

# Multi-step query requiring at least 2 tool calls
query = "Look up the current weather in London and Tokyo. Then tell me which city is warmer and by how many degrees."

result = agent.run(query)

[INFO] Agent starting with query: Look up the current weather in London and Tokyo. Then tell me which city is warmer and by how many degrees.

  [INFO] Iteration 1 of 10
  [INFO] Sending request to LLM...
  [INFO] LLM responded (finish_reason: tool_calls)
  [INFO] Model called 2 tool(s)
  [INFO]   Executing: get_weather({'city': 'London'})

Executing tool: get_weather
     Input: {'city': 'London'}
     Result: Weather in London: 12° Celsius, Rainy
  [INFO]   ✓ Tool returned: Weather in London: 12° Celsius, Rainy...
  [INFO]   Executing: get_weather({'city': 'Tokyo'})

Executing tool: get_weather
     Input: {'city': 'Tokyo'}
     Result: Weather in Tokyo: 18° Celsius, Partly Cloudy
  [INFO]   ✓ Tool returned: Weather in Tokyo: 18° Celsius, Partly Cloudy...
    [INFO] Iteration 2 of 10
    [INFO] Sending request to LLM...
    [INFO] LLM responded (finish_reason: tool_calls)
    [INFO] Model called 1 tool(s)
    [INFO]   Executing: calculator({'operand_a': 18, 'operand_b': 12, 'operatio

## Section 5: Memory & State Handling

### Conversation Memory vs. Working Memory

**Conversation Memory (Message History):**
- The complete history of messages exchanged with the LLM
- Used by LLM to maintain context across turns
- Example: all previous user queries and assistant responses
- Grows with each agent iteration

**Working Memory (Scratchpad/State):**
- Temporary state the agent tracks during task execution
- Information about what the agent has observed so far
- Intermediate results, reasoning steps, task progress
- Agent's internal "notebook" for organizing thoughts

### Example:
```
Conversation Memory:
  [User] "Compare weather in 2 cities"
  [Assistant] "I'll look up the weather..."
  [User] "London weather: 12°C, Rainy"
  [Assistant] "Now checking Tokyo..."
  
Working Memory (Agent's State):
  {
    "task": "Compare weather in London and Tokyo",
    "london_temp": 12,
    "tokyo_temp": 18,
    "cities_checked": 2,
    "task_complete": False
  }
```

In [23]:
# Enhanced Agent with explicit state tracking
class AgentWithStateTracking:
    """Agent that explicitly tracks conversation memory and working memory."""
    
    def __init__(self, max_iterations: int = 10, model: str = None):
        self.max_iterations = max_iterations
        self.model = model or AGENT_MODEL
        self.conversation_memory = []  # All messages exchanged with LLM
        self.working_memory = {         # Agent's internal state
            "observations": [],
            "reasoning_steps": [],
            "tool_calls_made": []
        }
    
    def log_reasoning(self, step: str):
        """Log a reasoning step to working memory."""
        self.working_memory["reasoning_steps"].append(step)
        print(f"Reasoning: {step}")
    
    def log_observation(self, observation: str):
        """Log an observation to working memory."""
        self.working_memory["observations"].append(observation)
        print(f"Observation: {observation}")
    
    def log_tool_call(self, tool_name: str, args: dict, result: str):
        """Log tool execution to working memory."""
        self.working_memory["tool_calls_made"].append({
            "tool": tool_name,
            "args": args,
            "result": result
        })
        print(f"Tool: {tool_name}({args}) → {result[:50]}...")
    
    def print_state_summary(self):
        """Print current working memory state."""
        print("\n" + "="*60)
        print("WORKING MEMORY STATE SUMMARY")
        print("="*60)
        print(f"Observations ({len(self.working_memory['observations'])}):")
        for obs in self.working_memory["observations"]:
            print(f"  • {obs}")
        
        print(f"\nReasoning Steps ({len(self.working_memory['reasoning_steps'])}):")
        for step in self.working_memory["reasoning_steps"]:
            print(f"  • {step}")
        
        print(f"\nTool Calls ({len(self.working_memory['tool_calls_made'])}):")
        for call in self.working_memory["tool_calls_made"]:
            print(f"  • {call['tool']} → {call['result'][:40]}...")
        print("="*60 + "\n")
    
    def run(self, user_query: str) -> str:
        """Run agent with explicit memory tracking."""
        print(f"\nStarting agent with query: {user_query}\n")
        
        self.conversation_memory.append({"role": "user", "content": user_query})
        self.log_reasoning("Task received: " + user_query)
        
        iteration = 0
        while iteration < self.max_iterations:
            iteration += 1
            print(f"\n--- Iteration {iteration} ---")
            
            try:
                # Send to LLM
                self.log_reasoning(f"Sending message to LLM (history length: {len(self.conversation_memory)})")
                
                response = client.chat.completions.create(
                    model=self.model,
                    messages=self.conversation_memory,
                    tools=tools,
                    tool_choice="auto",
                    temperature=0.7,
                    max_tokens=1024
                )
                
                assistant_message = response.choices[0].message
                
                # Check for tool calls
                if hasattr(assistant_message, 'tool_calls') and assistant_message.tool_calls:
                    self.log_reasoning(f"LLM decided to call {len(assistant_message.tool_calls)} tool(s)")
                    
                    self.conversation_memory.append({
                        "role": "assistant",
                        "content": assistant_message.content or ""
                    })
                    
                    for tool_call in assistant_message.tool_calls:
                        tool_name = tool_call.function.name
                        tool_input = json.loads(tool_call.function.arguments)
                        
                        # Execute tool
                        result = execute_tool(tool_name, tool_input)
                        
                        # Log to working memory
                        self.log_tool_call(tool_name, tool_input, result)
                        self.log_observation(f"{tool_name} returned: {result[:60]}")
                        
                        # Add to conversation memory
                        self.conversation_memory.append({
                            "role": "user",
                            "content": f"Tool {tool_name} result: {result}"
                        })
                
                else:
                    # Final answer
                    final_answer = assistant_message.content or ""
                    self.log_reasoning("LLM finished. No more tools needed.")
                    self.log_observation(f"Final answer: {final_answer[:80]}")
                    
                    self.print_state_summary()
                    
                    print("\nFINAL ANSWER:")
                    print(final_answer)
                    return final_answer
                    
            except Exception as e:
                print(f"Error: {str(e)}")
                return f"Agent failed: {str(e)}"
        
        return "Agent exceeded max iterations"

print("AgentWithStateTracking class defined")

AgentWithStateTracking class defined


In [24]:
# Test enhanced agent with state tracking
agent_with_memory = AgentWithStateTracking(max_iterations=10)
result = agent_with_memory.run("What is 15 plus 8? Then multiply the result by 2.")


Starting agent with query: What is 15 plus 8? Then multiply the result by 2.

Reasoning: Task received: What is 15 plus 8? Then multiply the result by 2.

--- Iteration 1 ---
Reasoning: Sending message to LLM (history length: 1)
Reasoning: LLM decided to call 2 tool(s)

Executing tool: calculator
     Input: {'operand_a': 15, 'operand_b': 8, 'operation': 'add'}
     Result: 15 add 8 = 23
Tool: calculator({'operand_a': 15, 'operand_b': 8, 'operation': 'add'}) → 15 add 8 = 23...
Observation: calculator returned: 15 add 8 = 23

Executing tool: calculator
     Input: {'operand_a': 23, 'operand_b': 2, 'operation': 'multiply'}
     Result: 23 multiply 2 = 46
Tool: calculator({'operand_a': 23, 'operand_b': 2, 'operation': 'multiply'}) → 23 multiply 2 = 46...
Observation: calculator returned: 23 multiply 2 = 46

--- Iteration 2 ---
Reasoning: Sending message to LLM (history length: 4)
Reasoning: LLM decided to call 2 tool(s)

Executing tool: calculator
     Input: {'operand_a': 15, 'operand_b

## Section 6: Failure Modes & Guardrails

### Deliberately Breaking the Agent

Test edge cases: ambiguous requests, error-returning tools, undefined tool calls, and more.

In [25]:
# Test Case 1: Ambiguous Request
print("\n" + "="*60)
print("TEST 1: AMBIGUOUS REQUEST")
print("="*60)
print("Query: 'Do something interesting'\n")

agent1 = MinimalAgent(max_iterations=5, debug=False)
result1 = agent1.run("Do something interesting for me.")

# Test Case 2: Tool that returns an error
print("\n" + "="*60)
print("TEST 2: TOOL ERROR (Division by Zero)")
print("="*60)
print("Query: 'What is 10 divided by 0?'\n")

agent2 = MinimalAgent(max_iterations=5, debug=False)
result2 = agent2.run("What is 10 divided by 0? Handle this gracefully.")

# Test Case 3: Undefined/Unknown Tool Request
print("\n" + "="*60)
print("TEST 3: UNDEFINED TOOL REQUEST")
print("="*60)
print("Query: 'Use the send_email tool to send me a message'\n")

agent3 = MinimalAgent(max_iterations=5, debug=False)
result3 = agent3.run("I want you to send an email using a special send_email function. Can you do that?")

print("\n" + "="*60)


TEST 1: AMBIGUOUS REQUEST
Query: 'Do something interesting'



Executing tool: calculator
     Input: {'operand_a': 5, 'operand_b': 7, 'operation': 'add'}
     Result: 5 add 7 = 12

Executing tool: read_file
     Input: {'file_path': 'interesting_facts.txt'}
     Result: Error: File not found: interesting_facts.txt

Executing tool: calculator
     Input: {'operand_a': 12, 'operand_b': 2, 'operation': 'multiply'}
     Result: 12 multiply 2 = 24

Executing tool: read_file
     Input: {'file_path': 'fun_facts.txt'}
     Result: Error: File not found: fun_facts.txt

Executing tool: calculator
     Input: {'operand_a': 24, 'operand_b': 10, 'operation': 'add'}
     Result: 24 add 10 = 34

Executing tool: read_file
     Input: {'file_path': 'jokes.txt'}
     Result: Error: File not found: jokes.txt

Executing tool: calculator
     Input: {'operand_a': 34, 'operand_b': 2, 'operation': 'multiply'}
     Result: 34 multiply 2 = 68

Executing tool: calculator
     Input: {'operand_a': 68, 'operan

### Observed Failure Modes & Mitigations

#### 1. **Infinite Loops**
- **Description**: Agent keeps calling tools in a cycle without making progress toward goal
- **Cause**: LLM doesn't recognize when it has enough information to answer
- **Mitigation**: 
  - `max_iterations` safeguard (implemented in MinimalAgent)
  - Add "tool call limit per tool type" constraint
  - Track which tools have been called recently; penalize repeating same tool

#### 2. **Hallucinated Tool Calls**
- **Description**: Model invents tool names or parameters that don't exist in schema
- **Cause**: LLM trained on data with tools it no longer has access to
- **Mitigation**:
  - Validate tool_name against available tools before executing
  - Return clear error message when tool doesn't exist
  - Add "Available tools: [list]" to system message

#### 3. **Wrong Tool Arguments (Malformed Input)**
- **Description**: Model calls correct tool but with invalid/malformed arguments
- **Cause**: JSON parsing failures, missing required fields, wrong types
- **Mitigation**:
  - Wrap tool calls in try-except and return detailed error message
  - Include example usage in tool descriptions
  - Use strict JSON schema validation

#### 4. **Silent Errors**
- **Description**: Tool fails but agent doesn't report it clearly; LLM doesn't understand what went wrong
- **Cause**: Generic error messages; no logging of failures
- **Mitigation**:
  - Log every tool call AND its result
  - Return explicit error context: "Error: X failed because Y"
  - Implement state tracking (working_memory) to track failed attempts

#### 5. **Ambiguous Requests → Endless Reasoning**
- **Description**: User query is vague; agent keeps asking for clarification in its reasoning but never calls tools
- **Cause**: Model's internal reasoning doesn't translate to tool calls
- **Mitigation**:
  - Add "If query is ambiguous, ask user for clarification" instruction
  - Track "tool_calls_made > 0" to ensure agent makes progress
  - Timeout on reasoning-only iterations (force tool call or answer)

#### 6. **Token Limit Exceeded**
- **Description**: Agent's message history grows so large it hits LLM context limit
- **Cause**: Long loops, many tool calls, verbose results
- **Mitigation**:
  - Implement "message history compression" (summarize old messages)
  - Store only last N messages
  - Separate "conversation_memory" (for context) from "working_memory" (scratchpad)

### Why Frameworks Like LangChain, LangGraph, and CrewAI Exist

Building raw agent loops reveals why these frameworks are valuable:

1. **State Management Complexity**: Managing conversation memory + working memory + tool results gets complicated. Frameworks provide structured state machines (LangGraph).

2. **Error Handling Boilerplate**: Every agent needs try-catch, logging, retries, malformed input handling. Frameworks bake these in.

3. **Tool Integration Simplicity**: Defining tools with JSON schemas is repetitive. Frameworks auto-generate schemas from Python functions.

4. **Multi-Agent Orchestration**: Coordinating multiple agents, passing results between them requires routing logic. CrewAI and LangGraph simplify this.

5. **Debugging & Observability**: Raw loops make it hard to trace what went wrong. Frameworks log each step, visualize graph execution, and integrate with observability platforms.

6. **Best Practices Enforcement**: Frameworks encourage max_iterations, timeout handling, structured outputs, etc.

**But knowing how to build this by hand is critical**: You understand what the framework is doing behind the scenes. When debugging a LangChain agent, you know to check message history, tool schemas, and iteration counts.

## Section 7: Multi-Step Agent Testing

Comprehensive test of the agent on complex multi-step tasks requiring correct tool sequencing, state persistence, and accurate final output.

In [26]:
print("\n" + "="*70)
print("MULTI-STEP TEST 1: Weather Comparison")
print("="*70)
print("Task: Look up weather in London, Tokyo, and Dubai. Rank by temperature.\n")

agent_test1 = AgentWithStateTracking(max_iterations=10)
result_test1 = agent_test1.run(
    "Look up the weather in London, Tokyo, and Dubai. "
    "Then rank all three cities from warmest to coldest and explain the differences."
)

print("\n" + "="*70)
print("MULTI-STEP TEST 2: Arithmetic Chain")
print("="*70)
print("Task: Calculate (50 - 20) * 3 + 15, then divide by 5\n")

agent_test2 = AgentWithStateTracking(max_iterations=10)
result_test2 = agent_test2.run(
    "Calculate: First subtract 20 from 50. "
    "Then multiply the result by 3. "
    "Then add 15 to that. "
    "Finally, divide the total by 5 and give me the final answer."
)

print("\n" + "="*70)
print("MULTI-STEP TEST 3: Mixed Tools")
print("="*70)
print("Task: Compare weather AND do calculations\n")

agent_test3 = AgentWithStateTracking(max_iterations=10)
result_test3 = agent_test3.run(
    "Get the weather in New York and Los Angeles. "
    "Convert both temperatures to Fahrenheit (if not already). "
    "Then calculate the difference and tell me which is warmer."
)


MULTI-STEP TEST 1: Weather Comparison
Task: Look up weather in London, Tokyo, and Dubai. Rank by temperature.


Starting agent with query: Look up the weather in London, Tokyo, and Dubai. Then rank all three cities from warmest to coldest and explain the differences.

Reasoning: Task received: Look up the weather in London, Tokyo, and Dubai. Then rank all three cities from warmest to coldest and explain the differences.

--- Iteration 1 ---
Reasoning: Sending message to LLM (history length: 1)
Reasoning: LLM decided to call 3 tool(s)

Executing tool: get_weather
     Input: {'city': 'London'}
     Result: Weather in London: 12° Celsius, Rainy
Tool: get_weather({'city': 'London'}) → Weather in London: 12° Celsius, Rainy...
Observation: get_weather returned: Weather in London: 12° Celsius, Rainy

Executing tool: get_weather
     Input: {'city': 'Tokyo'}
     Result: Weather in Tokyo: 18° Celsius, Partly Cloudy
Tool: get_weather({'city': 'Tokyo'}) → Weather in Tokyo: 18° Celsius, Partly 

## Summary: What You've Built

### The ReAct Loop in Action

You've implemented a **minimal but complete agent** that:
1. **Reasons**: Sends context to LLM, which decides what to do next
2. **Acts**: Executes the chosen tool with proper error handling
3. **Observes**: Captures results and appends them to message history
4. **Repeats**: Until the task is complete

### Key Takeaways

1. **Agents are just loops**: The "magic" is straightforward: LLM → parse response → execute tools → repeat
2. **Tool descriptions matter**: Clear, detailed tool descriptions help the LLM choose the right tool
3. **State tracking is essential**: Separating "conversation memory" (for LLM context) from "working memory" (agent's internal state) keeps logic clean
4. **Failure modes are predictable**: Infinite loops, hallucinated tools, malformed inputs—all solvable with simple guardrails
5. **Frameworks are conveniences, not magic**: LangChain, LangGraph, and CrewAI solve real problems (state management, error handling, multi-agent coordination), but understanding the raw loop is critical for debugging

### Next Steps (Week 5 Day 2)

Tomorrow you'll learn how LangChain and LangGraph abstract these patterns into:
- **LangChain**: Simple agent builders with tool integration
- **LangGraph**: State machines for complex multi-step workflows
- **CrewAI**: Multi-agent orchestration and role-based teams

But you've now seen what's happening under the hood. You can build agents. You can debug them. You understand what "agentic" really means.

---

**Notebook Completed**: Week 5 Day 1 - Agent Foundations
**Date**: 2026-08-12
**Framework**: Groq API (Claude-compatible)
**Key Concepts Covered**: ReAct pattern, tool calling, agent loops, memory management, failure modes